In [1]:
import numpy as np  #numpy: https://numpy.org/doc/stable/index.html
import importlib.util  #importlib: https://docs.python.org/3/library/importlib.html
from scipy.spatial import cKDTree
from astropy.table import Table
from numpy.lib import recfunctions as rfn
import matplotlib
import matplotlib.pyplot as plt
import time
import voronoi

def ltime(date=False):
    tm=time.localtime()
    datestr=str(tm.tm_mday)+'/'+str(tm.tm_mon)+'/'+str(tm.tm_year)
    tmstr=str(tm.tm_hour)+':'+str(tm.tm_min)+':'+str(tm.tm_sec)

    if(date):
        tmstr=datestr+' '+tmstr

    return tmstr

import os, sys
root = "/mnt/home/project/csatsangi.chetan/projects"
sys.path.insert(0, os.path.join(root, "sahyadri-sandbox", "scripts", "post-process"))

In [2]:
#Load data:
centrals = Table.read('centrals_sim.fits').as_array()
satellites = Table.read('satellites_sim.fits').as_array()
centrals_trimmed = rfn.drop_fields(centrals, ('Nsat','rvir'), usemask=False)
all_galaxies = np.concatenate([centrals_trimmed, satellites])

gpos = np.zeros((all_galaxies.shape[0], 3), dtype='float')
gpos[:,0] = all_galaxies['x']
gpos[:,1] = all_galaxies['y']
gpos[:,2] = all_galaxies['z']

#Parameters:
Ntrc = gpos.shape[0]
Lbox = 200.0
seed = 10
percentiles = np.arange(1, 100)
gpos = gpos%Lbox

## We have the data loaded. Now all we have to do is compute the vvf for different jackknife realizations
1. Store different jacnkknife realizations's positions
2. find their vvf for ranfac=10,000 parallelly
3. find the covarianvce matrix 

In [14]:
gpos.shape

(339777, 3)

In [4]:
gpos_jn,_ = add_pbc_jncol(gpos, rand=None, Lbox=Lbox, njn=125, los=1)
gpos_jn.shape

(339777, 4)

In [6]:
from jackknife import JackKnife
JackIt =  JackKnife(gpos, Lbox)
gpos_jn2,_ = JackIt.add_jackknife_regions(njn=125, los=1)
gpos_jn2.shape

(339777, 4)